# 15 — Manual Agentic GRPO: env → weights → RLCluster → GRPOLearner

Эта тетрадка не прячет обучение за `scripts/run_agentic_grpo.py`. Она вручную собирает все детали, чтобы можно было инспектировать каждый слой:

1. project/profile/config;
2. CrafText runtime и agentic environment;
3. Tunix topology/workload/preflight;
4. Qwen tokenizer/weights assets;
5. `RLCluster(actor, rollout, reference)`;
6. `ToolAgent`, `CrafTextAgenticEnvironment`, `GRPOLearner`;
7. `learner.train(...)`, checkpoints и evidence paths.

Важно: это GRPO notebook, а не PPO notebook. Он не использует `HybridPpoStep`, потому что в GRPO нет trainable critic/value boundary. `HybridPpoStep` остаётся для notebooks 10/11/12 и Agentic PPO/PPO-Lag/CPO lane. Здесь правильный production boundary — Tunix `TrajectoryCollectEngine` + `GRPOLearner` поверх `RLCluster`.

Локально можно выполнить profile/runtime/preflight и deterministic scripted group check без загрузки модели. Heavy ячейки отделены явно: readiness validation → загрузка весов → `RLCluster` → `GRPOLearner` → train.


## 0. External preparation

Перед real run подготовьте зависимости и локальные веса вне notebook:

```bash
pyenv exec python -m uv sync --extra envs --extra prompts --extra tunix --extra dev
# Qwen snapshot должен существовать локально:
# artifacts/models/qwen25-05b-instruct
```

Notebook не скачивает веса неявно.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import jax

from tunix_craftext.env.agentic_craftext import (
    CrafTextAgenticEnvironment,
    CrafTextStepTool,
    agentic_environment_kwargs,
)
from tunix_craftext.training.agentic_grpo_smoke import (
    collect_scripted_grpo_group_sync,
    save_scripted_grpo_smoke,
)
from tunix_craftext.env.config import load_mvp_config
from tunix_craftext.training.grpo_profile import build_grpo_evidence_manifest, load_agentic_grpo_profile
from tunix_craftext.tunix.preflight import pinned_qwen_tensor_shape, validate_agentic_grpo_preflight
from tunix_craftext.tunix.rlcluster_workload import (
    AgenticGrpoWorkloadSpec,
    build_agentic_grpo_cluster,
    load_agentic_grpo_qwen_assets,
)
from tunix_craftext.env.runtime import build_craftext_runtime
from tunix_craftext.env.tasks import CrafTextTaskSampler
from tunix_craftext.models.tunix_adapter import load_qwen_hf_tokenizer
from tunix_craftext.tunix.topology import load_tunix_topology, role_to_meshes


In [ ]:
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Run this notebook from inside the tunix-craftext repository')

PROFILE_PATH = ROOT / 'configs/grpo/qwen_agentic_local.yaml'
SCRIPTED_EVIDENCE = ROOT / 'artifacts/runs/agentic-grpo-scripted-smoke-notebook.json'

print('repo:', ROOT)
print('jax backend:', jax.default_backend())


## 1. Load profile and MVP config manually

Profile describes the model/topology/workload/evidence paths. MVP config builds the real CrafText runtime.


In [ ]:
profile = load_agentic_grpo_profile(PROFILE_PATH)
config_path = ROOT / profile.environment_config
topology_path = ROOT / profile.topology_config
snapshot = ROOT / profile.model.snapshot

config = load_mvp_config(config_path)
runtime = build_craftext_runtime(config)
task_sampler = CrafTextTaskSampler.from_runtime(
    runtime,
    horizon=config.environment.horizon,
    mode='cycle',
    fixed_instruction_index=config.environment.instruction_index,
    goal_prefix=profile.run.goal,
)
prepared_goal, prepared_instruction_index = task_sampler.task_at(config.environment.instruction_index)

print('profile:', profile.run.name)
print('goal:', profile.run.goal)
print('environment config:', config_path, config_path.is_file())
print('topology config:', topology_path, topology_path.is_file())
print('snapshot:', snapshot, snapshot.is_dir())
print('action count:', runtime.action_count)
print('first actions:', runtime.actions.labels[:8])
print('first prepared CrafText task:', prepared_instruction_index, prepared_goal)


## 2. Build and inspect one real CrafText agentic environment

This manually constructs the same environment class that Tunix workers use. It proves config → runtime → prompt/tool environment before touching weights.


In [ ]:
env_kwargs = agentic_environment_kwargs(config_path)
task = {
    'goal': prepared_goal,
    'seed': config.run.seed,
    'horizon': config.environment.horizon,
    'instruction_index': prepared_instruction_index,
}
manual_env = CrafTextAgenticEnvironment(task, **env_kwargs, group_id=0, pair_index=0)
observation, info = manual_env.reset()

print('env kwargs:', env_kwargs)
print('reset info:', info)
print('initial prompt preview:')
print(observation['question'][:1000])


## 3. Topology, workload spec and preflight

This is the no-weight allocation gate. It validates role meshes and static batch/cache constraints.


In [ ]:
topology = load_tunix_topology(topology_path)
meshes = role_to_meshes(topology)
spec = profile.workload
validate_agentic_grpo_preflight(topology, spec, pinned_qwen_tensor_shape())

print('topology:', topology.name, topology.role_to_device_indices)
for role, mesh in meshes.items():
    print(role, mesh.shape, mesh.axis_names)
print('workload:', spec)


## 4. Evidence manifest before model allocation

This records profile SHA, model identity, package versions and artifact paths before model allocation.


In [ ]:
manifest = build_grpo_evidence_manifest(profile, profile_path=PROFILE_PATH, repo_root=ROOT)
profile.evidence.provenance.parent.mkdir(parents=True, exist_ok=True)
profile.evidence.provenance.write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('wrote:', profile.evidence.provenance)
print(json.dumps(manifest['workload'], indent=2, sort_keys=True))


## 5. Deterministic GRPO group check: same env/tool loop, no model

This runs several generations for one task group through the same CrafText tool-call environment, but uses deterministic scripted tool calls instead of LLM generation. It proves agentic environment semantics, action validation, group rewards and evidence serialization before real model allocation.


In [ ]:
action_sequences = tuple(
    tuple('NOOP' if generation % 2 == 0 else 'LEFT' for _ in range(config.environment.horizon))
    for generation in range(profile.workload.num_generations)
)
scripted_results = collect_scripted_grpo_group_sync(
    config_path=config_path,
    goal=prepared_goal,
    seed=config.run.seed,
    group_id=0,
    action_sequences=action_sequences,
    horizon=config.environment.horizon,
)
save_scripted_grpo_smoke(SCRIPTED_EVIDENCE, scripted_results)
for result in scripted_results:
    print(result)
print('saved:', SCRIPTED_EVIDENCE)


## 6. Validate readiness for real weights

This is the manual stop line before heavy model allocation. It checks local snapshot, accelerator backend and evidence/checkpoint directories. If this cell passes, continue with the next cells deliberately.


In [ ]:
# Heavy readiness boundary.
# This cell performs validation only. If it passes, run the next cells manually to allocate weights.
if not snapshot.is_dir():
    raise FileNotFoundError(f'Local Qwen snapshot is required: {snapshot}')
if jax.default_backend() == 'cpu':
    raise RuntimeError(
        'Real Agentic Qwen GRPO is intended for an accelerator runner. '
        'Stop here on CPU; if you deliberately want to reproduce/debug CPU behavior, '
        'edit this readiness cell explicitly before continuing.'
    )

for evidence_path in (
    profile.evidence.root,
    profile.evidence.checkpoints,
    profile.evidence.trajectories.parent,
    profile.evidence.metrics.parent,
):
    evidence_path.mkdir(parents=True, exist_ok=True)

print('ready for real Qwen asset loading')
print('snapshot:', snapshot)
print('backend:', jax.default_backend())
print('checkpoint root:', profile.evidence.checkpoints)
print('metrics path:', profile.evidence.metrics)
print('trajectories path:', profile.evidence.trajectories)


## 7. Load tokenizer and Qwen actor/reference weights manually

This is the first heavy allocation cell. It loads actor/reference assets on declared role meshes and keeps tokenizer explicit.


In [ ]:
assets = load_agentic_grpo_qwen_assets(snapshot, topology)
hf_tokenizer = load_qwen_hf_tokenizer(snapshot)
print('loaded actor:', type(assets.actor))
print('loaded reference:', type(assets.reference))
print('loaded tokenizer:', type(assets.tokenizer))
print('loaded hf tokenizer:', type(hf_tokenizer))


## 8. Build RLCluster manually

No learner yet: this just creates the public Tunix cluster from explicit assets and config.


In [ ]:
cluster = build_agentic_grpo_cluster(topology, spec, assets)
print('cluster:', type(cluster))
print('roles:', {role.value: mesh.shape for role, mesh in cluster.cluster_config.role_to_mesh.items()})


## 9. Build ToolAgent + GRPOLearner manually

This cell constructs Tunix `GRPOLearner` directly from the already loaded assets and `RLCluster`. The learner owns actor/reference policy updates and checkpointing through Tunix training config; CrafText remains an agentic tool environment.


In [ ]:
from tunix.rl.agentic.agentic_grpo_learner import GRPOConfig, GRPOLearner
from tunix.rl.agentic.agents.tool_agent import ToolAgent
from tunix.rl.agentic.parser.chat_template_parser.parser import QwenChatTemplateParser

learner = GRPOLearner(
    rl_cluster=cluster,
    algo_config=GRPOConfig(
        num_generations=profile.workload.num_generations,
        max_response_length=profile.workload.max_new_tokens,
        max_concurrency=profile.workload.max_concurrency,
        system_prompt='Use craftext_step for every environment action.',
    ),
    chat_parser=QwenChatTemplateParser(hf_tokenizer, enable_thinking=False),
    agent_class=ToolAgent,
    agent_kwargs={
        'system_prompt': 'Use craftext_step for every environment action.',
        'tool_parser_name': 'qwen',
        'tool_map': {'craftext_step': CrafTextStepTool},
    },
    env_class=CrafTextAgenticEnvironment,
    env_kwargs=env_kwargs,
)
print('learner:', type(learner))


## 10. Train manually and close the cluster

This cell calls `learner.train(...)` directly. Run it only after inspecting readiness, loaded assets, cluster and learner cells above. After the call, inspect `profile.evidence.metrics`, `profile.evidence.trajectories` and `profile.evidence.checkpoints`.


In [ ]:
train_iterator = task_sampler.batches(
    seed=config.run.seed,
    batch_size=profile.workload.mini_batch_size,
    count=profile.workload.max_steps,
)
preview_batch = next(task_sampler.batches(
    seed=config.run.seed,
    batch_size=profile.workload.mini_batch_size,
    count=1,
))
print('training task preview:')
print(json.dumps({
    'goal': preview_batch['goal'],
    'seed': preview_batch['seed'].tolist(),
    'horizon': preview_batch['horizon'].tolist(),
    'instruction_index': preview_batch['instruction_index'].tolist(),
}, indent=2, ensure_ascii=False))

try:
    learner.train(train_iterator, skip_jit=False)
finally:
    cluster.close()


## 11. After GRPO: PPO-Lag / CPO extension

Once this GRPO path runs on target hardware, PPO-Lag/CPO should reuse the same agentic environment/tool transport and add:

- reward critic for PPO values;
- cost critic for safety/cost return;
- `HybridPpoStep`/Agentic PPO train examples for token logprobs, masks, values and policy version;
- Lagrange multiplier update for PPO-Lag;
- constrained objective / projection terms for CPO;
- checkpoint metadata for actor, reference, reward critic, cost critic, optimizer and policy version.
